# Orchestrator-Workers Workflow

## 소개

같은 작업에 여러 관점이 필요했지만 어떤 관점이 가장 가치 있을지 미리 예측할 수 없었던 적이 있나요? orchestrator-workers 패턴은 중앙 LLM이 각각의 고유한 작업을 분석하고 특화된 worker LLM에 위임할 최적의 하위 작업을 동적으로 결정하도록 하여 이를 해결합니다.

기존 접근 방식은 여러 번 수동으로 프롬프트를 작성해야 하거나, 맥락과 관계없이 동일한 변형을 생성하는 하드코딩된 parallelization을 사용합니다.

이 접근 방식에서는 orchestrator LLM이 작업을 분석하고 이 특정 사례에서 어떤 변형이 가장 가치 있을지 결정한 뒤, 각 변형을 생성하는 worker LLM에 위임합니다.

### 만들게 될 것

제품 설명 요청을 받아 다음을 수행하는 시스템입니다.

1. 어떤 유형의 marketing copy가 가치 있을지 분석합니다.
2. worker를 위한 특화된 작업 설명을 동적으로 생성합니다.
3. 서로 다른 대상 독자에 최적화된 여러 콘텐츠 변형을 만듭니다.
4. 모든 worker의 결과를 조율해 반환합니다.

### 사전 요구 사항

- Python 3.9 이상
- 환경 변수로 설정한 Anthropic API key: `export ANTHROPIC_API_KEY='your-key'`
- prompt engineering의 기본 이해
- Python class와 type hint에 대한 지식


### 이 workflow를 사용할 때

이 workflow는 필요한 하위 작업을 미리 예측할 수 없는 복잡한 작업에 적합합니다. 단순한 parallelization과의 핵심 차이는 유연성입니다. 하위 작업을 미리 정의하지 않고 orchestrator가 구체적인 입력을 바탕으로 결정합니다.

**이 패턴을 사용할 때:**

- 작업에 서로 다른 여러 접근 방식이나 관점이 필요합니다.
- 최적의 하위 작업이 구체적인 입력에 따라 달라집니다.
- 서로 다른 전략이나 스타일을 비교해야 합니다.

**이 패턴을 사용하지 않을 때:**

- 단일 결과를 내는 간단한 작업입니다(불필요한 복잡성).
- latency가 중요합니다(여러 LLM 호출로 overhead가 추가됨).
- 하위 작업을 예측하고 미리 정의할 수 있습니다(더 간단한 parallelization 사용).

## 작동 방식

orchestrator-workers 패턴은 두 단계로 작동합니다.

1. **분석 및 계획 단계**: orchestrator LLM이 작업과 맥락을 받아 어떤 접근 방식이 가치 있을지 분석하고 XML 형식으로 구조화된 하위 작업 설명을 생성합니다.

2. **실행 단계**: 각 worker LLM은 다음을 받습니다.
   - 맥락을 위한 원래 작업
   - 자신에게 주어진 구체적인 하위 작업 유형과 설명
   - 제공된 추가 맥락

orchestrator는 어떤 하위 작업을 만들지 *runtime에* 결정하므로 미리 정의된 parallel workflow보다 더 적응적입니다.

## 설정

### 설치
```bash
pip install anthropic
```

### Helper 함수
이 예제는 LLM 호출과 XML 응답 parsing에 `util.py`의 helper 함수를 사용합니다.

- `llm_call(prompt, system_prompt="", model="claude-sonnet-4-6")`: Claude에 프롬프트를 보내고 텍스트 응답을 반환합니다.
- `extract_xml(text, tag)`: regex를 사용해 XML tag에서 내용을 추출합니다.

이 utility들은 API 인증(환경에서 `ANTHROPIC_API_KEY` 읽기)을 처리하고 orchestrator-workers 패턴을 위한 간단한 interface를 제공합니다. 전체 구현은 [util.py](util.py)에서 볼 수 있습니다.

## 구현

`FlexibleOrchestrator` class는 두 단계 workflow를 조율합니다.

**핵심 설계 결정:**
- 프롬프트는 유연성을 위해 runtime 변수(`task`, `context`)를 받는 template입니다.
- 구조화된 출력 parsing에 XML을 사용합니다(안정적이며 language model에 친화적인 형식).
- 더 나은 맥락을 위해 worker는 원래 작업과 자신의 구체적인 지시를 모두 받습니다.
- error handling에서 worker가 비어 있지 않은 응답을 반환하는지 검증합니다.

구현에 포함되는 것:
- `parse_tasks()`: orchestrator의 XML 출력을 구조화된 작업 dictionary로 parse합니다.
- `FlexibleOrchestrator.process()`: orchestrator를 호출한 뒤 worker들을 호출하는 핵심 조율 로직입니다.
- 비어 있는 worker 출력을 감지하고 처리하는 응답 검증

In [ ]:
from util import extract_xml, llm_call

# Model 설정
MODEL = "claude-sonnet-4-6"  # orchestrator와 worker 모두에 빠르고 성능 좋은 model 사용


def parse_tasks(tasks_xml: str) -> list[dict]:
    """XML 작업을 작업 dictionary 목록으로 parse합니다."""
    tasks = []
    current_task = {}

    for line in tasks_xml.split("\n"):
        line = line.strip()
        if not line:
            continue

        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<type>"):
            current_task["type"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[12:-13].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)

    return tasks


class FlexibleOrchestrator:
    """작업을 분해하고 worker LLM을 사용해 병렬로 실행합니다."""

    def __init__(
        self,
        orchestrator_prompt: str,
        worker_prompt: str,
        model: str = MODEL,
    ):
        """프롬프트 template과 model 선택으로 초기화합니다."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt
        self.model = model

    def _format_prompt(self, template: str, **kwargs) -> str:
        """변수를 사용해 프롬프트 template의 형식을 지정합니다."""
        try:
            return template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}") from e

    def process(self, task: str, context: dict | None = None) -> dict:
        """작업을 분해하고 하위 작업을 병렬 실행하여 처리합니다."""
        context = context or {}

        # 1단계: orchestrator 응답 받기
        orchestrator_input = self._format_prompt(self.orchestrator_prompt, task=task, **context)
        orchestrator_response = llm_call(orchestrator_input, model=self.model)

        # orchestrator 응답 parse
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)

        print("\n" + "=" * 80)
        print("ORCHESTRATOR ANALYSIS")
        print("=" * 80)
        print(f"\n{analysis}\n")

        print("\n" + "=" * 80)
        print(f"IDENTIFIED {len(tasks)} APPROACHES")
        print("=" * 80)
        for i, task_info in enumerate(tasks, 1):
            print(f"\n{i}. {task_info['type'].upper()}")
            print(f"   {task_info['description']}")

        print("\n" + "=" * 80)
        print("GENERATING CONTENT")
        print("=" * 80 + "\n")

        # 2단계: 각 작업 처리
        worker_results = []
        for i, task_info in enumerate(tasks, 1):
            print(f"[{i}/{len(tasks)}] Processing: {task_info['type']}...")

            worker_input = self._format_prompt(
                self.worker_prompt,
                original_task=task,
                task_type=task_info["type"],
                task_description=task_info["description"],
                **context,
            )

            worker_response = llm_call(worker_input, model=self.model)
            worker_content = extract_xml(worker_response, "response")

            # worker 응답 검증 - 빈 출력 처리
            if not worker_content or not worker_content.strip():
                print(f"⚠️  Warning: Worker '{task_info['type']}' returned no content")
                worker_content = f"[Error: Worker '{task_info['type']}' failed to generate content]"

            worker_results.append(
                {
                    "type": task_info["type"],
                    "description": task_info["description"],
                    "result": worker_content,
                }
            )

        # 결과 표시
        print("\n" + "=" * 80)
        print("RESULTS")
        print("=" * 80)
        for i, result in enumerate(worker_results, 1):
            print(f"\n{'-' * 80}")
            print(f"Approach {i}: {result['type'].upper()}")
            print(f"{'-' * 80}")
            print(f"\n{result['result']}\n")

        return {
            "analysis": analysis,
            "worker_results": worker_results,
        }

## 사용 사례 예제: Marketing 변형 생성

이제 실제적인 예제로 orchestrator-workers 패턴이 작동하는 모습을 보겠습니다. 제품을 위한 여러 스타일의 marketing copy를 생성합니다.

**이 예제가 패턴을 잘 보여 주는 이유:**
- 제품에 따라 효과적인 marketing 관점이 다릅니다.
- "최선의" 변형은 구체적인 제품 특성과 대상 독자에 따라 달라집니다.
- orchestrator는 고정된 template 대신 입력에 따라 전략을 조정할 수 있습니다.

**프롬프트 설계 참고 사항:**
- orchestrator 프롬프트는 2~3개의 접근 방식을 요청하고 XML 구조 지침을 제공합니다.
- worker 프롬프트는 worker에게 전체 맥락(원래 작업, 자신의 스타일, 지침)을 제공합니다.
- 두 프롬프트 모두 안정적인 parsing을 위해 명확한 XML 형식을 사용합니다.

In [9]:
ORCHESTRATOR_PROMPT = """
Analyze this task and break it down into 2-3 distinct approaches:

Task: {task}

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
    <type>formal</type>
    <description>Write a precise, technical version that emphasizes specifications</description>
    </task>
    <task>
    <type>conversational</type>
    <description>Write an engaging, friendly version that connects with readers</description>
    </task>
</tasks>
"""

WORKER_PROMPT = """
Generate content based on:
Task: {original_task}
Style: {task_type}
Guidelines: {task_description}

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"""


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task="Write a product description for a new eco-friendly water bottle",
    context={
        "target_audience": "environmentally conscious millennials",
        "key_features": ["plastic-free", "insulated", "lifetime warranty"],
    },
)


ORCHESTRATOR ANALYSIS


This task requires creating marketing copy for an eco-friendly water bottle. The core challenge is balancing product information with persuasive messaging while highlighting the environmental benefits. Different approaches would serve distinct marketing channels and audience segments:

1. A **feature-focused technical approach** would appeal to detail-oriented consumers who make purchasing decisions based on specifications, materials, and measurable environmental impact. This serves e-commerce listings and comparison shopping.

2. A **lifestyle-oriented emotional approach** would connect with values-driven consumers through storytelling and aspirational messaging, emphasizing how the product fits into an eco-conscious lifestyle. This serves social media and brand-building content.

3. A **benefit-driven practical approach** would focus on solving everyday problems while weaving in sustainability advantages, appealing to mainstream consumers who want both functi

## 요약

이제 구체적인 입력에 따라 작업 분해를 동적으로 조정하는 orchestrator-workers 패턴을 구현했습니다. 이 패턴은 어떤 변형을 만들지 미리 정의할 필요 없이 여러 marketing copy 변형을 생성했으며, 각 변형은 서로 다른 대상 독자와 맥락에 맞춰졌습니다.

### 핵심 요점

**패턴의 이점:**
- **적응성**: orchestrator가 각각의 고유한 입력에 최선의 접근 방식을 결정합니다.
- **유연성**: 프롬프트를 바꿔 다른 도메인에 쉽게 적용할 수 있습니다.
- **구조화된 조율**: XML 기반 통신이 안정적인 parsing을 보장합니다.
- **오류에 대한 복원력**: 검증으로 worker 실패를 감지하고 처리합니다.

**이 패턴이 뛰어난 경우:**
- 여러 관점이 필요한 콘텐츠 생성(marketing, 문서 작성, 창작)
- 서로 다른 분석 관점이 도움이 되는 분석 작업
- 분해 전략이 문제에 따라 달라지는 문제 해결

### 제약 및 고려 사항

**비용과 Latency:**
- N+1회의 LLM 호출 필요(orchestrator 1회 + worker N회)
- 이 구현에서는 순차 처리(worker가 한 번에 하나씩 실행됨)
- 더 나은 성능을 위해 `asyncio` 또는 thread pool로 worker 호출을 병렬화하는 것을 고려하세요.

**이 패턴을 사용하지 말아야 할 때:**
- 단일하고 명확한 결과를 내는 간단한 작업(추가 복잡성을 정당화하기 어려움)
- latency가 중요한 application(여러 API 호출로 overhead 추가)
- 하위 작업이 항상 동일한 작업(미리 정의된 parallelization 사용)

**고려할 실패 양상:**
- orchestrator가 작업을 최적으로 분해하지 못할 수 있습니다(prompt engineering이 중요함).
- worker가 비어 있거나 잘못된 형식의 응답을 반환할 수 있습니다(검증으로 처리함).
- model이 형식을 정확히 따르지 않으면 XML parsing이 실패할 수 있습니다(대안으로 JSON 사용 고려).

### 다음 단계

**이 구현 개선하기:**
1. 더 나은 성능을 위해 `asyncio`로 worker 병렬 실행 추가
2. 실패한 worker에 대한 retry 로직 구현
3. LLM이 worker 결과를 결합하는 synthesis 단계 추가
4. 서로 다른 orchestrator 전략 실험(예: 더 많거나 적은 하위 작업 요청)

**자신의 사용 사례에 맞게 조정하기:**
- 도메인에 맞는 작업 분해를 유도하도록 orchestrator 프롬프트 수정
- 도메인별 지침을 제공하도록 worker 프롬프트 조정
- application에 관련된 context parameter 추가
- 비용과 품질을 최적화하기 위해 orchestrator에는 Claude Opus, worker에는 Claude Haiku 사용 고려